$$\{t_i,\ y_i,\ \sigma_i\}_{i=1}^{N},\qquad N = 28$$

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.linalg import solve_triangular

rng = np.random.default_rng(2)
def truth(t): return 10 + 0.04*t + 0.9*np.sin(2*np.pi*t/9 + 0.6)
t = np.sort(np.concatenate([rng.uniform(0, 17, 16), rng.uniform(27, 40, 12)]))
N = t.size
sigma = rng.uniform(0.15, 0.35, N)
bad = rng.choice(N, 5, replace=False)
sigma[bad] = rng.uniform(0.6, 0.9, bad.size)
y = truth(t) + sigma*rng.normal(size=N)
tt = np.linspace(-2, 42, 300)
C = np.diag(sigma**2)

plt.errorbar(t, y, sigma, fmt='ko')
plt.plot(tt, truth(tt), 'k:')
plt.xlabel('t [d]'); plt.ylabel('y [mJy]')
plt.show()

$$\mathbf f = A\mathbf w,\qquad \mathbf w \sim \mathcal N(\mathbf 0,\ \Lambda) \quad\Rightarrow\quad \mathbf f \sim \mathcal N(\mathbf 0,\ A\Lambda A^\top)$$

$$\mathrm{Cov}\big[f(t), f(t')\big] = \mathbf a(t)^\top\Lambda\,\mathbf a(t') \equiv k(t, t')$$

$$\text{line: } \mathbf a(t) = \begin{bmatrix} 1\\ s\end{bmatrix},\quad s = \frac{t - 20}{20},\quad \Lambda = \begin{bmatrix} \lambda_b^2 & 0\\ 0 & \lambda_m^2\end{bmatrix}:\qquad k(t, t') = \lambda_b^2 + \lambda_m^2\, s\, s'$$

In [ ]:
def a(t): return np.column_stack([np.ones_like(t), (t - 20)/20])
Lam = np.diag([1.0, 1.0])

plt.plot(tt, a(tt) @ rng.multivariate_normal([0, 0], Lam, 20).T, 'C0', alpha=0.4)
# plt.axvline(25, color='r'); 
# plt.axvline(30, color='r')
plt.xlabel('t [d]'); plt.ylabel('f(t)')
plt.show()

$$\begin{bmatrix} f(25)\\ f(30)\end{bmatrix} \sim \mathcal N\!\left(\begin{bmatrix} 0\\ 0\end{bmatrix},\ \begin{bmatrix} k_{11} & k_{12}\\ k_{21} & k_{22}\end{bmatrix}\right)
\quad\Rightarrow\quad
p\big(f(30)\mid f(25) = a\big) = \mathcal N\!\left(\frac{k_{21}}{k_{11}}\,a,\ \ k_{22} - \frac{k_{21}^2}{k_{11}}\right)$$

$$\begin{bmatrix} \mathbf x_1\\ \mathbf x_2\end{bmatrix} \sim \mathcal N\!\left(\mathbf 0,\ \begin{bmatrix} \Sigma_{11} & \Sigma_{12}\\ \Sigma_{21} & \Sigma_{22}\end{bmatrix}\right)
\quad\Rightarrow\quad
\boxed{\ \boldsymbol\mu_{2|1} = \Sigma_{21}\Sigma_{11}^{-1}\mathbf x_1,\qquad \Sigma_{2|1} = \Sigma_{22} - \Sigma_{21}\Sigma_{11}^{-1}\Sigma_{12}\ }$$

In [ ]:
S = a(np.array([25.0, 30.0])) @ Lam @ a(np.array([25.0, 30.0])).T
x = rng.multivariate_normal([0, 0], S, 100000)
a0 = 1.5
keep = np.abs(x[:, 0] - a0) < 0.05
mu_c = S[1, 0]/S[0, 0]*a0
sd_c = np.sqrt(S[1, 1] - S[1, 0]**2/S[0, 0])
print('ρ =', round(S[0, 1]/np.sqrt(S[0, 0]*S[1, 1]), 3), '   f(30) | f(25) = 1.5:', round(mu_c, 2), '±', round(sd_c, 2))

plt.plot(x[:5000, 0], x[:5000, 1], '.', ms=2, alpha=0.2)
plt.plot(x[keep, 0][:150], x[keep, 1][:150], 'r.', ms=3)
plt.errorbar([a0], [mu_c], [sd_c], fmt='ko', capsize=6)
plt.xlabel('f(25)'); plt.ylabel('f(30)'); plt.axis('equal')
plt.show()

$$\mathbf y = \mathbf f + \boldsymbol\varepsilon,\qquad \mathbf f \sim \mathcal N(\mathbf 0, K),\quad \boldsymbol\varepsilon \sim \mathcal N(\mathbf 0, C)
\quad\Rightarrow\quad
\begin{bmatrix} \mathbf y\\ \mathbf f_*\end{bmatrix} \sim \mathcal N\!\left(\mathbf 0,\ \begin{bmatrix} K + C & K_*\\ K_*^\top & K_{**}\end{bmatrix}\right)$$

$$K = k(\mathbf t, \mathbf t),\qquad K_* = k(\mathbf t, \mathbf t_*),\qquad K_{**} = k(\mathbf t_*, \mathbf t_*)$$

$$\boxed{\ \boldsymbol\mu_* = K_*^\top(K + C)^{-1}\mathbf y,\qquad \Sigma_* = K_{**} - K_*^\top(K + C)^{-1}K_*\ }$$

$$\mathbf y \to \mathbf y - \bar y$$

In [ ]:
ybar = np.average(y, weights=1/sigma**2)
y0 = y - ybar

def gp_predict(k, t_new):
    K, K_s, K_ss = k(t, t), k(t, t_new), k(t_new, t_new)
    mu = K_s.T @ np.linalg.solve(K + C, y0)
    cov = K_ss - K_s.T @ np.linalg.solve(K + C, K_s)
    return mu, np.sqrt(np.clip(np.diag(cov), 0, None))

A_line = np.column_stack([np.ones(N), t])
w_hat = np.linalg.solve(A_line.T @ np.linalg.solve(C, A_line), A_line.T @ np.linalg.solve(C, y))

def k_lin(t1, t2, lam=10.0):
    return lam**2 * (1 + np.multiply.outer((t1 - 20)/20, (t2 - 20)/20))

mu, sd = gp_predict(k_lin, tt)
plt.errorbar(t, y0, sigma, fmt='ko')
plt.plot(tt, mu); plt.fill_between(tt, mu - 2*sd, mu + 2*sd, alpha=0.3)
plt.plot(tt, w_hat[0] + w_hat[1]*tt - ybar, 'r--')
plt.xlabel('t [d]'); plt.ylabel('y − ȳ')
plt.show()

$$f \sim \mathcal{GP}(0,\ k) \quad\Longleftrightarrow\quad \begin{bmatrix} f(t_1)\\ \vdots\\ f(t_n)\end{bmatrix} \sim \mathcal N(\mathbf 0, K),\quad K_{ij} = k(t_i, t_j)\quad \text{for any } t_1, \ldots, t_n$$

$$\tau = t - t'$$

$$k_{\rm SE} = A^2 e^{-\tau^2/2\ell^2}\qquad k_{\rm M32} = A^2\Big(1 + \tfrac{\sqrt3|\tau|}{\ell}\Big)e^{-\sqrt3|\tau|/\ell}$$

$$k_{\rm per} = A^2\exp\!\Big[-\Gamma\sin^2\tfrac{\pi\tau}{P}\Big]\qquad k_{\rm QP} = A^2\exp\!\Big[-\tfrac{\tau^2}{2\ell^2} - \Gamma\sin^2\tfrac{\pi\tau}{P}\Big]$$

In [ ]:
def k_SE(t1, t2, A=0.8, ell=3.0):
    tau = np.subtract.outer(t1, t2); return A**2*np.exp(-tau**2/(2*ell**2))
def k_M32(t1, t2, A=0.8, ell=3.0):
    x = np.sqrt(3)*np.abs(np.subtract.outer(t1, t2))/ell; return A**2*(1 + x)*np.exp(-x)
def k_per(t1, t2, A=0.8, P=9.0, G=2.0):
    tau = np.subtract.outer(t1, t2); return A**2*np.exp(-G*np.sin(np.pi*tau/P)**2)
def k_QP(t1, t2, A=0.8, ell=20.0, P=9.0, G=2.0):
    tau = np.subtract.outer(t1, t2); return A**2*np.exp(-tau**2/(2*ell**2) - G*np.sin(np.pi*tau/P)**2)

tau = np.linspace(0, 30, 400); zero = np.zeros(1)
for name, k in [('SE', k_SE), ('Matérn-3/2', k_M32), ('periodic', k_per), ('quasi-periodic', k_QP)]:
    plt.plot(tau, k(tau, zero)[:, 0]/0.8**2, label=name)
plt.legend(); plt.xlabel('τ [d]'); plt.ylabel('k(τ) / A²')
plt.show()

$$K = LL^\top,\qquad \mathbf z \sim \mathcal N(\mathbf 0, I)\quad\Rightarrow\quad \mathbf f = L\mathbf z \sim \mathcal N(\mathbf 0,\ LL^\top) = \mathcal N(\mathbf 0, K)$$

In [ ]:
tg = np.linspace(0, 40, 400)
for k, color in [(k_SE, 'C0'), (k_per, 'C1')]:
    L = np.linalg.cholesky(k(tg, tg) + 1e-8*np.eye(tg.size))
    plt.plot(tg, L @ rng.normal(size=(tg.size, 2)), color)
plt.xlabel('t [d]'); plt.ylabel('f(t)')
plt.show()

$$k = k_{\rm SE}:\quad A = 0.8,\ \ \ell = 3\ {\rm d}$$

In [ ]:
mu, sd = gp_predict(k_SE, tt)
plt.errorbar(t, y0, sigma, fmt='ko')
plt.plot(tt, mu); plt.fill_between(tt, mu - 2*sd, mu + 2*sd, alpha=0.3)
plt.plot(tt, truth(tt) - ybar, 'k:')
plt.xlabel('t [d]'); plt.ylabel('y − ȳ')
plt.show()

$$k = k_{\rm SE}:\quad A = 0.8,\qquad \ell = 0.7,\ 3,\ 15\ {\rm d}$$

In [ ]:
for ell, color in [(0.7, 'C0'), (3.0, 'C1'), (15.0, 'C2')]:
    mu, sd = gp_predict(lambda t1, t2: k_SE(t1, t2, 0.8, ell), tt)
    plt.plot(tt, mu, color, label=f'ℓ = {ell} d')
    plt.fill_between(tt, mu - 2*sd, mu + 2*sd, color=color, alpha=0.12)
plt.errorbar(t, y0, sigma, fmt='ko')
plt.legend(); plt.xlabel('t [d]'); plt.ylabel('y − ȳ')
plt.show()

$$k = k_{\rm QP}:\quad A = 0.8,\ \ \ell = 20\ {\rm d},\ \ P = 9\ {\rm d},\ \ \Gamma = 2$$

In [ ]:
mu, sd = gp_predict(k_QP, tt)
plt.errorbar(t, y0, sigma, fmt='ko')
plt.plot(tt, mu); plt.fill_between(tt, mu - 2*sd, mu + 2*sd, alpha=0.3)
plt.plot(tt, truth(tt) - ybar, 'k:')
plt.xlabel('t [d]'); plt.ylabel('y − ȳ')
plt.show()

$$p(\mathbf y \mid \boldsymbol\theta) = \int p(\mathbf y \mid \mathbf f)\,p(\mathbf f \mid \boldsymbol\theta)\,d\mathbf f = \mathcal N(\mathbf y \mid \mathbf 0,\ K_\theta + C)$$

$$\ln p(\mathbf y \mid \boldsymbol\theta) = \underbrace{-\tfrac12\,\mathbf y^\top(K_\theta + C)^{-1}\mathbf y}_{\text{data fit}}\ \underbrace{-\ \tfrac12\ln|K_\theta + C|}_{\text{complexity}}\ -\ \tfrac N2\ln 2\pi$$

In [ ]:
def ln_p_parts(k):
    M = k(t, t) + C
    fit = -0.5*y0 @ np.linalg.solve(M, y0)
    complexity = -0.5*np.linalg.slogdet(M)[1]
    return fit, complexity, fit + complexity - 0.5*N*np.log(2*np.pi)

ells = np.geomspace(0.3, 60, 120)
parts = np.array([ln_p_parts(lambda t1, t2: k_SE(t1, t2, 0.8, ell)) for ell in ells])
plt.semilogx(ells, parts[:, 0], label='data fit')
plt.semilogx(ells, parts[:, 1], label='complexity')
plt.semilogx(ells, parts[:, 2], 'k', label='ln p')
plt.legend(); plt.xlabel('ℓ [d]  (SE kernel)')
plt.show()

$$\hat{\boldsymbol\theta} = \arg\max_{\boldsymbol\theta}\ \ln p(\mathbf y \mid \boldsymbol\theta),\qquad \boldsymbol\theta = (A,\ \ell,\ \Gamma,\ P)$$

In [ ]:
def neg_ln_p(theta):
    A_, ell_, G_, P_ = np.exp(theta[0]), np.exp(theta[1]), np.exp(theta[2]), theta[3]
    return -ln_p_parts(lambda t1, t2: k_QP(t1, t2, A_, ell_, P_, G_))[2]

res = minimize(neg_ln_p, [np.log(0.8), np.log(20), np.log(2), 10.0], method='L-BFGS-B',
               bounds=[(-3, 2), (np.log(3), np.log(200)), (-3, 3), (5, 15)])
A_fit, ell_fit, G_fit, P_fit = np.exp(res.x[0]), np.exp(res.x[1]), np.exp(res.x[2]), res.x[3]
print(f'A = {A_fit:.2f},  ℓ = {ell_fit:.1f} d,  Γ = {G_fit:.2f},  P = {P_fit:.2f} d,  ln p = {-res.fun:.2f}')

k_fit = lambda t1, t2: k_QP(t1, t2, A_fit, ell_fit, P_fit, G_fit)
mu, sd = gp_predict(k_fit, tt)
plt.errorbar(t, y0, sigma, fmt='ko')
plt.plot(tt, mu); plt.fill_between(tt, mu - 2*sd, mu + 2*sd, alpha=0.3)
plt.plot(tt, truth(tt) - ybar, 'k:')
plt.xlabel('t [d]'); plt.ylabel('y − ȳ')
plt.show()

$$K + C = LL^\top$$

$$\mathbf z = L^{-1}\mathbf y:\qquad \mathbf y^\top(K + C)^{-1}\mathbf y = \mathbf z^\top\mathbf z,\qquad \ln|K + C| = 2\sum_i\ln L_{ii}$$

$$\text{cost} \propto N^3,\qquad \text{memory} \propto N^2$$

In [ ]:
M = k_fit(t, t) + C
L = np.linalg.cholesky(M)
z = solve_triangular(L, y0, lower=True)
print('z·z =', round(z @ z, 6), '   yᵀ(K+C)⁻¹y =', round(y0 @ np.linalg.solve(M, y0), 6))
print('2 Σ ln L_ii =', round(2*np.log(np.diag(L)).sum(), 6), '   ln|K+C| =', round(np.linalg.slogdet(M)[1], 6))

rng_t = np.random.default_rng(4)
Ns, secs = [250, 500, 1000, 2000, 4000], []
for n in Ns:
    x_ = np.sort(rng_t.uniform(0, n/10, n)); M_ = k_SE(x_, x_, 1.0, 3.0) + 0.1*np.eye(n)
    t0 = time.perf_counter(); np.linalg.cholesky(M_); secs.append(time.perf_counter() - t0)
plt.loglog(Ns, secs, 'o-', label='Cholesky')
plt.loglog(Ns, secs[-1]*(np.array(Ns)/Ns[-1])**3, 'k--', label='∝ N³')
plt.legend(); plt.xlabel('N'); plt.ylabel('seconds')
plt.show()

$$\underbrace{\mathcal N(\mathbf y \mid A\mathbf w,\ C)}_{\text{likelihood}}\ \underbrace{\mathcal N(\mathbf w \mid \mathbf 0,\ \Lambda)}_{\text{prior}} = \underbrace{\mathcal N(\mathbf w \mid \hat{\mathbf w},\ \Sigma_w)}_{\text{posterior}}\ \underbrace{\mathcal N(\mathbf y \mid \mathbf 0,\ C + A\Lambda A^\top)}_{\text{marginal likelihood}},\qquad \Sigma_w^{-1} = \Lambda^{-1} + A^\top C^{-1}A$$

$$\boxed{\ (C + A\Lambda A^\top)^{-1} = C^{-1} - C^{-1}A\,\Sigma_w\,A^\top C^{-1}\ },\qquad |C + A\Lambda A^\top| = |C|\,|\Lambda|\,|\Sigma_w^{-1}|$$

$$N\times N \ \to\ M\times M$$

In [ ]:
def ln_p_dense(Ab, lam2, s2, yv):
    M_ = (Ab*lam2) @ Ab.T + np.diag(s2)
    return -0.5*yv @ np.linalg.solve(M_, yv) - 0.5*np.linalg.slogdet(M_)[1] - 0.5*len(yv)*np.log(2*np.pi)

def ln_p_woodbury(Ab, lam2, s2, yv):
    Cy, CA = yv/s2, Ab/s2[:, None]
    Sw_inv = np.diag(1/lam2) + Ab.T @ CA
    u = Ab.T @ Cy
    quad = yv @ Cy - u @ np.linalg.solve(Sw_inv, u)
    logdet = np.linalg.slogdet(Sw_inv)[1] + np.sum(np.log(lam2)) + np.sum(np.log(s2))
    return -0.5*quad - 0.5*logdet - 0.5*len(yv)*np.log(2*np.pi)

centres = np.arange(0, 400, 8.0)
def bumps(x_): return np.exp(-0.5*((x_[:, None] - centres[None, :])/6)**2)
Mb = centres.size
sd_, sw_ = [], []
for n in Ns:
    x_ = np.sort(rng_t.uniform(0, 400, n)); Ab = bumps(x_); s2 = np.full(n, 0.1)
    yv = Ab @ rng_t.normal(size=Mb) + np.sqrt(s2)*rng_t.normal(size=n)
    t0 = time.perf_counter(); d = ln_p_dense(Ab, np.ones(Mb), s2, yv); sd_.append(time.perf_counter() - t0)
    t0 = time.perf_counter(); w = ln_p_woodbury(Ab, np.ones(Mb), s2, yv); sw_.append(time.perf_counter() - t0)
print(f'N = {Ns[-1]}, M = {Mb}:   dense ln p = {d:.6f}   Woodbury ln p = {w:.6f}')
plt.loglog(Ns, sd_, 'o-', label='dense  (N × N)')
plt.loglog(Ns, sw_, 'o-', label=f'Woodbury  (M × M, M = {Mb})')
plt.legend(); plt.xlabel('N'); plt.ylabel('seconds')
plt.show()

$$\frac{d^2f}{dt^2} + \frac{\omega_0}{Q}\frac{df}{dt} + \omega_0^2 f = \text{noise}
\quad\Rightarrow\quad
k_{\rm SHO}(\tau) = S_0\,\omega_0 Q\,e^{-\omega_0\tau/2Q}\Big[\cos(\eta\,\omega_0\tau) + \tfrac{\sin(\eta\,\omega_0\tau)}{2\eta Q}\Big],\quad \eta = \sqrt{1 - \tfrac{1}{4Q^2}}$$

$$t_1 < t_2 < \cdots < t_N:\qquad e^{-c\,(t_n - t_m)} = e^{-c\,t_n}\,e^{c\,t_m}\quad\Rightarrow\quad \text{cost} \propto N$$

In [ ]:
import celerite2
from celerite2 import terms

sho = terms.SHOTerm(sigma=0.8, rho=9.0, Q=5.0)
gp = celerite2.GaussianProcess(sho, mean=0.0)
gp.compute(t, yerr=sigma)
M_sho = sho.get_value(np.subtract.outer(t, t)) + C
dense = -0.5*y0 @ np.linalg.solve(M_sho, y0) - 0.5*np.linalg.slogdet(M_sho)[1] - 0.5*N*np.log(2*np.pi)
print(f'celerite ln p = {gp.log_likelihood(y0):.8f}    dense ln p = {dense:.8f}')

Nc, sc = [1000, 3000, 10000, 30000, 100000], []
for n in Nc:
    x_ = np.sort(rng_t.uniform(0, n/50, n)); yv = rng_t.normal(size=n)
    t0 = time.perf_counter(); gp.compute(x_, yerr=0.3); gp.log_likelihood(yv); sc.append(time.perf_counter() - t0)
plt.loglog(Ns, secs, 'o-', label='dense Cholesky')
plt.loglog(Nc, sc, 'o-', label='celerite')
plt.legend(); plt.xlabel('N'); plt.ylabel('seconds')
plt.show()

$$\mathbf y \sim \mathcal N\big(a + b\,\mathbf t,\ K_{\rm SHO} + C\big),\qquad \tau_{\rm coh} = \frac{2Q}{\omega_0} = \frac{QP}{\pi}$$

In [ ]:
def sho_gp(p):
    ln_s, P_, ln_Q, a_, b_ = p
    g = celerite2.GaussianProcess(terms.SHOTerm(sigma=np.exp(ln_s), rho=P_, Q=np.exp(ln_Q)), mean=lambda x_: a_ + b_*x_)
    g.compute(t, yerr=sigma)
    return g

res_c = min((minimize(lambda p: -sho_gp(p).log_likelihood(y0), [np.log(0.8), P0, np.log(5), 0.0, 0.0], method='L-BFGS-B',
                      bounds=[(-3, 2), (5, 15), (np.log(0.6), 6), (-3, 3), (-0.5, 0.5)]) for P0 in [7, 9, 11]),
            key=lambda r: r.fun)
print(f'P = {res_c.x[1]:.2f} d,  Q = {np.exp(res_c.x[2]):.0f},  a = {res_c.x[3]:.2f},  b = {res_c.x[4]:.3f}')

mu, var = sho_gp(res_c.x).predict(y0, t=tt, return_var=True)
plt.errorbar(t, y0, sigma, fmt='ko')
plt.plot(tt, mu); plt.fill_between(tt, mu - 2*np.sqrt(var), mu + 2*np.sqrt(var), alpha=0.3)
plt.plot(tt, res_c.x[3] + res_c.x[4]*tt, 'r--')
plt.plot(tt, truth(tt) - ybar, 'k:')
plt.xlabel('t [d]'); plt.ylabel('y − ȳ')
plt.show()

$$\frac{\partial\ln p}{\partial\theta_j} = \tfrac12\,\boldsymbol\alpha^\top\frac{\partial K}{\partial\theta_j}\boldsymbol\alpha - \tfrac12\,\mathrm{tr}\!\Big[(K + C)^{-1}\frac{\partial K}{\partial\theta_j}\Big],\qquad \boldsymbol\alpha = (K + C)^{-1}\mathbf y$$

In [ ]:
import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp
from tinygp import GaussianProcess, kernels

def neg_ln_p_tg(theta):
    k = jnp.exp(2*theta[0]) * kernels.ExpSquared(scale=jnp.exp(theta[1])) * kernels.ExpSineSquared(scale=theta[3], gamma=jnp.exp(theta[2]))
    return -GaussianProcess(k, t, diag=sigma**2).log_probability(y0)

f_and_g, f_only = jax.jit(jax.value_and_grad(neg_ln_p_tg)), jax.jit(neg_ln_p_tg)
hist_g, hist_f = [], []
def with_grad(th):
    v, gr = f_and_g(th); hist_g.append(float(v)); return float(v), np.asarray(gr)
def without_grad(th):
    v = f_only(th); hist_f.append(float(v)); return float(v)
th0 = np.array([np.log(0.8), np.log(20), np.log(2), 10.0])
bnds = [(-3, 2), (np.log(3), np.log(200)), (-3, 3), (5, 15)]
minimize(with_grad, th0, jac=True, method='L-BFGS-B', bounds=bnds)
minimize(without_grad, th0, method='L-BFGS-B', bounds=bnds)
best = min(hist_g + hist_f)
plt.semilogy(np.array(hist_g) - best + 1, label='JAX gradient')
plt.semilogy(np.array(hist_f) - best + 1, label='finite differences')
plt.legend(); plt.xlabel('ln p evaluations'); plt.ylabel('−ln p − min + 1')
plt.show()